In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy import linalg
import networkx as nx # 用于生成复杂的异质拓扑
from scipy.stats import ortho_group
from scipy import stats

In [2]:
def init_matrix_with_controlled_spectrum(N, distribution='uniform', bounds=(0.1, 1.0), symmetric=True):
    """
    初始化权重矩阵 W，并强制其精度矩阵 A = WW^T 具有特定的谱分布和增强的非对角相关性。

    Args:
        N (int): 维度。
        distribution (str): 谱分布类型 ('uniform', 'power_law', 'bimodal')。
        bounds (tuple): 特征值范围 (min, max)。
        symmetric (bool): True=对称W, False=非对称W (Dale兼容)。
        align_mean_mode (bool): 
            - True (关键修改): 强制第一个特征向量指向 [1,1...1]。这会将 lambda[0] 的能量
              广播到整个矩阵的非对角元，使 A_ij 的量级不再趋于 0。
            - False: 纯随机旋转，A_ij 通常为 O(1/sqrt(N))，远小于 A_ii。

    Returns:
        W (ndarray): 权重矩阵。
        A (ndarray): 精度矩阵。
    """
    # --- 1. 生成 A 的特征向量基 Q (Output Basis) ---

    # 原始逻辑：纯随机基 (导致 A 接近对角阵)
    if N < 1000:
        Q = ortho_group.rvs(N)
    else:
        H = np.random.randn(N, N)
        Q, _ = linalg.qr(H)

    # --- 2. 定义 A 的目标谱 (Eigenvalues) ---
    if distribution == 'orthogonal':
        lambdas = np.ones(N) # A = I
    
    elif distribution == 'uniform':
        low, high = bounds
        lambdas = np.random.uniform(low, high, N)
        
    elif distribution == 'power_law':
        # 幂律分布 (Sloppy Model 常用)
        alpha = 1.0
        x = np.random.uniform(0, 1, N)
        lambdas = (1 - x) ** (-1/alpha)
        # 映射到 bounds
        l_min, l_max = lambdas.min(), lambdas.max()
        lambdas = bounds[0] + (lambdas - l_min) * (bounds[1] - bounds[0]) / (l_max - l_min)
        
    elif distribution == 'bimodal':
        # 双峰分布：模拟 E-I 分离的强相互作用
        # 一半很强(Stiff)，一半很弱(Sloppy)
        mid = N // 2
        lambdas = np.zeros(N)
        lambdas[:mid] = np.random.uniform(bounds[0], bounds[0]*10, mid) # Sloppy
        lambdas[mid:] = np.random.uniform(bounds[1]/10, bounds[1], N-mid) # Stiff
        
    else:
        raise ValueError("Unknown distribution")


    # --- 3. 重构 W (Reconstruction) ---
    # A = Q * diag(lambda) * Q.T
    
    sqrt_lam = np.sqrt(lambdas)
    
    if symmetric:
        # W 是对称平方根
        W = Q @ np.diag(sqrt_lam) @ Q.T
    else:
        # W 是混合符号 (Mixed Signs)
        # 使用独立的随机基 V 作为输入空间
        if N < 1000:
            V = ortho_group.rvs(N)
        else:
            H_v = np.random.randn(N, N)
            V, _ = linalg.qr(H_v)
            
        W = Q @ np.diag(sqrt_lam) @ V.T
    
    return W

In [3]:
class RobustHeterogeneousNetwork:
    def __init__(self, N, topology='scale_free', avg_degree=6, 
                 alpha=0.05, C=100.0, 
                 lambda_smooth=0.01, drop_prob=0.1):
        """
        Args:
            N (int): 神经元数量
            topology (str): 'small_world', 'scale_free', 'random'
            avg_degree (int): 平均连接度
            alpha (float): 熵正则化系数 (防止秩崩溃)
            C (float): 总能量约束 (Trace constraint)
            lambda_smooth (float): 拉普拉斯平滑强度 (强制空间连续性，降低IPR)
            drop_prob (float): DropConnect 概率 (引入噪声，强制鲁棒性)
        """
        self.N = N
        self.alpha = alpha
        self.C = C
        self.lambda_smooth = lambda_smooth
        self.drop_prob = drop_prob
        
        # 1. 构建异质拓扑结构 (Adjacency Mask)
        self.mask = self._generate_topology(topology, avg_degree)
        
        # 2. 预计算结构拉普拉斯矩阵 (Structural Laplacian)
        # 用于正则化：L = D - Adj
        # 这是基于物理连接的，不是基于功能权重的
        degree_matrix = np.diag(np.sum(self.mask, axis=1))
        self.L_struct = degree_matrix - self.mask
        
        # 3. 初始化权重 (Random Orthogonal-ish)
        # 使用较小的随机值，打破对称性
        w=init_matrix_with_controlled_spectrum(N, distribution='uniform', bounds=(0.9, 1))
        # w=np.random.randn(N, N) * 0.1
        w*=self.mask
        self.W = w
        
        current_energy = np.sum(w** 2) + 1e-12
        scale = np.sqrt(self.C / current_energy)
        w*= scale
        self.W_init=w
        
        A_init=w@ w.T + np.eye(self.N)*1e-6
        self.A_init=A_init
        self.A=A_init

    def _generate_topology(self, kind, avg_k):
        if kind == 'small_world':
            G = nx.watts_strogatz_graph(self.N, k=avg_k, p=0.2)
        elif kind == 'scale_free':
            m = max(1, avg_k // 2)
            G = nx.barabasi_albert_graph(self.N, m)
        else: # Random
            p = avg_k / self.N
            G = nx.erdos_renyi_graph(self.N, p)
            
        adj = nx.to_numpy_array(G)
        np.fill_diagonal(adj, 1.0) # Self-loops allowed
        return adj

    def normalize_W(self):
        """严格执行能量约束: Tr(WW^T) = C"""
        current_energy = np.sum(self.W ** 2) + 1e-12
        scale = np.sqrt(self.C / current_energy)
        self.W *= scale

    def get_effective_W(self, training=False):
        """
        获取当前权重。
        如果 training=True，应用 DropConnect (随机屏蔽连接)。
        """
        W_masked = self.W * self.mask
        
        if training and self.drop_prob > 0: #drop connection 使 W最终不对称
            # 生成二值 Drop 掩码 (1 = 保留, 0 = 丢弃)
            # 概率 (1-p) 保留
            drop_mask = (np.random.rand(*self.W.shape) > self.drop_prob).astype(float)
            
            # 缩放权重以保持期望值不变 (Inverted Dropout)
            # W_eff = W * mask / (1-p)
            scale = 1.0 / (1.0 - self.drop_prob)
            return W_masked * drop_mask * scale
        
        return W_masked

    def update_hebbian(self, v, lr):
        """
        v: 输入信号在神经元上的投影 (Input projection)
        """
        # 1. 获取“带噪声”的权重视图 (DropConnect)
        # 这迫使网络不依赖特定的单一连接
        W_eff = self.get_effective_W(training=True)
        
        # 2. 计算当前精度矩阵 A (基于带噪声的权重)
        A = W_eff @ W_eff.T + np.eye(self.N)*1e-6
        self.A=A
        
        # 计算 A 的逆 (协方差)
        try:
            A_inv = np.linalg.inv(A)
        except np.linalg.LinAlgError:
            A_inv = np.linalg.pinv(A)

        # --- 梯度项计算 ---

        # A. Fisher Information Gradient (驱动力)
        # Term: v * v^T * W
        # 这一项让主要特征向量对齐信号方向
        grad_fisher = np.outer(v, v) @ W_eff
        
        # B. Entropy Regularization (最大熵/白化)
        # Term: alpha * A^-1 * W
        # 这一项防止所有特征值坍缩到同一个方向，保持秩
        grad_entropy = self.alpha * (A_inv @ W_eff)
        
        # C. Laplacian Regularization (空间平滑约束) - NEW
        # Loss = lambda * Tr(W^T L W) -> 邻居节点的权重应相似
        # Gradient = 2 * lambda * L @ W
        # 这强迫“软模式”在网络拓扑上是平滑变化的，直接对抗局域化(Localization)
        grad_smooth = self.lambda_smooth * (self.L_struct @ self.W)

        # D. Trace Constraint (能量约束拉格朗日乘子)
        # 动态计算 beta 以抵消增长
        v_proj = v.T @ A @ v
        # 粗略估计 beta，依靠后续 normalize_W 做精确修正
        beta = (v_proj + self.alpha * self.N) / self.C
        grad_trace = beta * W_eff


        
        
        # --- 组合梯度 ---
        # 注意：grad_smooth 是减去项（惩罚），grad_trace 也是减去项
        total_grad = 2 * (grad_fisher + grad_entropy - grad_trace) - grad_smooth
        
        # --- 更新 ---
        self.W += lr * total_grad
        
        # 强制拓扑约束 (Re-masking)
        self.W *= self.mask
        
        # 严格归一化 (防止数值漂移)
        self.normalize_W()

In [4]:
def get_gradients(N, mode='local'):
    x = np.linspace(0, 1, N)
    if mode == 'local':
        # 1. Localized: Only neurons near center care about s
        # return np.exp(-(x - 0.5)**2 / (2 * 0.05**2)).reshape(-1, 1)
        return np.sin(x * 0.5) * np.exp(-0.5 * (x / 15.0)**2) # High freq sensitivity
    
    elif mode == 'global_shift':
        # 2. Global: A wave-like gradient across all neurons
        return np.sin(2 * np.pi * x).reshape(-1, 1)
        # return np.exp(-0.5 * (x / 6.0)**2) # Low freq sensitivity
    
    elif mode == 'mean_field':
        # 3. Mean Field: All neurons care equally

        a=np.ones((N, 1))
        # a[:N//2]=-1
        return a#np.random.choice([1, -1], size=N)#np.ones((N, 1))
        # return np.linspace(0.9,1.0,N)

In [5]:
def calculate_xi_field_theory(A, N):
    """
    Corrected Ginzburg-Landau correlation length calculation.
    xi = sqrt(kappa / m^2)
    """
    # 1. Get eigenvalues and eigenvectors
    # We use eigh because A is symmetric
    vals, vecs = np.linalg.eigh(A)
    
    # Sort them just in case (eigh usually returns sorted)
    idx = np.argsort(vals)
    vals = vals[idx]
    vecs = vecs[:, idx]
    
    # 2. Identify the 'Mass' (Soft Mode eigenvalue)
    # This is the precision gap. If A is singular, m2 -> 0
    m2 = vals[0]
    
    # 4. Compute Correlation Length
    # Use a small epsilon to avoid division by zero at the critical point
    epsilon = 1e-12
    xi = np.sqrt(1 / (m2 + epsilon))
    
    # 5. Finite Size Clipping
    # Physical xi cannot meaningfully exceed the system size N
    return min(xi, N)

In [6]:
result_dir='result/'
modes=['local','global_shift','mean_field']
N_arr=np.array([400])
C = 2e2
H_weight=1e-3
topo='random'

In [7]:
A_arr=[]
W_arr=[]
Ai_arr=[]
Wi_arr=[]
for N in N_arr:
    f_prime = get_gradients(N, mode=modes[2])
    model=RobustHeterogeneousNetwork(N, topology=topo, avg_degree=N//10, alpha=H_weight, C=C, 
                 lambda_smooth=0.01, drop_prob=0.1)
    h_fisher, h_xi,h_xi_fft= [], [],[]
    for _ in range(300):
        W = model.get_effective_W()
        A =  W @ W.T + np.eye(N)*1e-6
        fi=f_prime.T @ A @ f_prime
        h_fisher.append(fi.item())
        
        h_xi.append(calculate_xi_field_theory(A, N))
        # h_xi_fft.append(calculate_xi_fourier_method(A, N))
                
        model.update_hebbian(f_prime, lr=0.001)
    
    A_arr.append(A)
    W_arr.append(model.W)
    Ai_arr.append(model.A_init)
    Wi_arr.append(model.W_init)


    


In [8]:
# --- Analysis & Power Law Plotting ---
def plot_power_law(data, label, ax, title, num_bins=None):
    """
    Plot power law distribution with adaptive binning.
    
    Parameters:
    - data: array-like, the data to plot
    - label: str, label for the data
    - ax: matplotlib axis, axis to plot on
    - title: str, title for the plot
    - num_bins: int or None, if None uses adaptive binning based on data size
    """
    # Adaptive binning based on data size
    if num_bins is None:
        # Use Freedman-Diaconis rule for bin width selection
        n = len(data)
        if n > 1000:
            # For large datasets, use more bins
            num_bins = int(np.sqrt(n))
        else:
            # For smaller datasets, use fewer bins
            num_bins = max(10, int(n / 50))
    
    # Ensure we have at least 5 bins and at most 50 bins
    num_bins = max(5, min(50, num_bins))
    
    # Log-spaced bins
    bins = np.logspace(np.log10(max(min(data), 1e-10)),  # Avoid log(0)
                      np.log10(max(data)), 
                      num_bins + 1)
    
    # Calculate histogram
    counts, bins = np.histogram(data, bins=bins)
    x = (bins[:-1] + bins[1:]) / 2
    y = counts / np.diff(bins)  # Probability density
    
    # Remove zeros for log-log fit
    mask = (y > 0) & (x > 0)
    x_fit, y_fit = x[mask], y[mask]
    
    # Check if we have enough points for fitting
    if len(x_fit) < 3:
        ax.loglog(x, y, 'o', label=f'{label} (insufficient data for fit)')
        ax.set_title(title)
        ax.set_xlabel('Value')
        ax.set_ylabel('Probability Density')
        ax.legend()
        return
    
    # Fit: log(y) = -tau * log(x) + C
    slope, intercept, r_value, p_value, std_err = linregress(np.log10(x_fit), np.log10(y_fit))
    
    # Plot data and fit
    ax.loglog(x, y, 'o', markersize=8, color='cyan',alpha=0.7, label=f'{label}')
    ax.loglog(x_fit, 10**intercept * x_fit**slope, '--', alpha=0.8, 
              linewidth=2,color='black', label=f'Fit: slope={slope:.2f}')
    
    # Add R² value if correlation is high
    if abs(r_value) > 0.8:
        ax.text(0.05, 0.95, f'R² = {r_value**2:.3f}', 
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top')
    
    # ax.set_title(title)
    ax.set_xlabel(label,fontsize=16, fontweight='bold')
    ax.set_ylabel('Probability Density',fontsize=16, fontweight='bold')
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3, which='both')

In [9]:
# --- Analysis & Power Law Plotting ---
def plot_power_law(data, label, ax, title, num_bins=None, color='#2E86AB'):
    """
    Plot power law distribution with adaptive binning.
    
    Parameters:
    - data: array-like, the data to plot
    - label: str, label for the data
    - ax: matplotlib axis, axis to plot on
    - title: str, title for the plot
    - num_bins: int or None, if None uses adaptive binning based on data size
    - color: str, color for the data points
    """
    # Set global style for the axis
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    
    # Adaptive binning based on data size
    if num_bins is None:
        n = len(data)
        if n > 1000:
            # Use Scott's rule for large datasets
            num_bins = int(np.ceil((3.5 * np.std(np.log10(data[data>0])) * n**(1/3))))
            num_bins = max(20, min(80, num_bins))
        else:
            # For smaller datasets, use Sturges' rule
            num_bins = int(np.ceil(np.log2(n) + 1))
            num_bins = max(10, min(30, num_bins))
    
    # Ensure we have reasonable number of bins
    num_bins = max(5, min(80, num_bins))
    
    # Clean data: remove zeros and negative values for log-scale
    data_clean = data[data > 0]
    if len(data_clean) == 0:
        ax.text(0.5, 0.5, 'No positive data points', 
                transform=ax.transAxes, ha='center', va='center')
        return
    
    # Log-spaced bins with protection against extreme values
    min_val = max(min(data_clean), 1e-10)
    max_val = max(data_clean)
    
    # Create logarithmically spaced bins
    bins = np.logspace(np.log10(min_val), np.log10(max_val), num_bins + 1)
    
    # Calculate histogram with density normalization
    counts, bin_edges = np.histogram(data_clean, bins=bins, density=True)
    x = (bin_edges[:-1] + bin_edges[1:]) / 2
    y = counts
    
    # Remove zeros for log-log fit
    mask = (y > 0) & (x > 0) & np.isfinite(y) & np.isfinite(x)
    x_fit, y_fit = x[mask], y[mask]
    
    # Check if we have enough points for fitting
    if len(x_fit) < 5:
        ax.loglog(x, y, 'o', markersize=7, color=color, alpha=0.6,
                 markerfacecolor='white', markeredgewidth=1.5,
                 label=f'{label} (insufficient data)')
        ax.set_xlabel(label, fontsize=14, fontweight='bold')
        ax.set_ylabel('Probability Density', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.2, linestyle=':', linewidth=0.8)
        ax.set_axisbelow(True)
        return
    
    # Fit: log(y) = -tau * log(x) + C
    try:
        slope, intercept, r_value, p_value, std_err = linregress(
            np.log10(x_fit), np.log10(y_fit)
        )
    except:
        ax.loglog(x, y, 'o', markersize=7, color=color, alpha=0.6,
                 markerfacecolor='white', markeredgewidth=1.5,
                 label=f'{label} (fit failed)')
        return
    
    # Color palette
    fit_color = '#D64933'  # Red for fit line
    
    # Plot data points with enhanced styling
    ax.loglog(x, y, 'o', 
              markersize=8,
              color=color,
              markerfacecolor='white',
              markeredgewidth=2,
              alpha=0.8,
              label=f'{label}',
              zorder=2)
    
    # Add error bars based on Poisson statistics (sqrt(count))
    y_err = np.sqrt(counts[mask]) / np.diff(bins)[mask] if len(mask) > 0 else None
    if y_err is not None and len(y_err) > 0:
        ax.errorbar(x_fit, y_fit, 
                   yerr=y_err,
                   fmt='none',
                   ecolor=color,
                   alpha=0.3,
                   capsize=2,
                   elinewidth=0.8,
                   zorder=1)
    
    # Plot fit line with enhanced styling
    x_fit_smooth = np.logspace(np.log10(min(x_fit)), np.log10(max(x_fit)), 100)
    y_fit_smooth = 10**intercept * x_fit_smooth**slope
    
    ax.loglog(x_fit_smooth, y_fit_smooth, 
              '-',
              color=fit_color,
              linewidth=2.5,
              alpha=0.85,
              label=f'$slope = {slope:.2f}$',
              zorder=3)
    
    # Add shaded region for fit uncertainty (optional)
    # y_fit_upper = 10**(intercept + std_err) * x_fit_smooth**slope
    # y_fit_lower = 10**(intercept - std_err) * x_fit_smooth**slope
    # ax.fill_between(x_fit_smooth, y_fit_lower, y_fit_upper,
    #                 color=fit_color, alpha=0.1, zorder=0)
    
    # Add R² and p-value in a clean box
    r_squared = r_value**2
    stats_text = f'$R^2 = {r_squared:.3f}$'
    # stats_text = f'$R^2 = {r_squared:.3f}$\n$p = {p_value:.2e}$'
    
    # Place statistics in top-left corner with a clean background
    bbox_props = dict(boxstyle='round,pad=0.4', 
                     facecolor='white', 
                     alpha=0.85,
                     edgecolor='gray',
                     linewidth=0.5)
    
    ax.text(0.05, 0.95, stats_text,
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=bbox_props,
            zorder=10)
    
    # # Add power law exponent annotation
    # exponent_text = f'$\\tau = {slope:.2f} \\pm {std_err:.3f}$'
    # ax.text(0.05, 0.82, exponent_text,
    #         transform=ax.transAxes,
    #         fontsize=10,
    #         verticalalignment='top',
    #         color=fit_color,
    #         fontweight='bold',
    #         bbox=bbox_props,
    #         zorder=10)
    
    # Axis labels with enhanced formatting
    ax.set_xlabel(label, fontsize=15, fontweight='bold', labelpad=8)
    ax.set_ylabel('Probability Density', fontsize=15, fontweight='bold', labelpad=8)
    
    # Legend with improved styling
    legend = ax.legend(fontsize=12,
                      frameon=True,
                      fancybox=True,
                      shadow=True,
                      framealpha=0.95,
                      edgecolor='none',
                      loc='lower left')
    legend.get_frame().set_linewidth(0)
    
    # Grid with subtle styling
    ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.8, which='both')
    ax.set_axisbelow(True)
    
    # Customize tick parameters
    ax.tick_params(axis='both', which='major', 
                   labelsize=11, width=1.5, length=6)
    ax.tick_params(axis='both', which='minor',
                   width=1, length=3)
    
    # Set axis limits with padding (MODIFIED: reduced padding on left)
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    
    # Get actual data range
    actual_x_min = np.min(x_fit)  # Use only the fitted data range
    actual_x_max = np.max(x_fit)
    actual_y_min = np.min(y_fit)
    actual_y_max = np.max(y_fit)
    
    # Set limits with asymmetric padding (less padding on left)
    x_padding_left = actual_x_min * 0.1  # Only 10% padding on left
    x_padding_right = actual_x_max * 7  # 20% padding on right
    y_padding_bottom = actual_y_min * 0.1
    y_padding_top = actual_y_max * 7
    
    ax.set_xlim(x_padding_left, x_padding_right)
    ax.set_ylim(y_padding_bottom, y_padding_top)
    
    # Optional: Add minor ticks
    ax.minorticks_on()
    
    return slope, intercept, r_value, p_value, std_err

In [10]:
# from scipy.integrate import trapz
def simulate_quench_avalanche(A, N, num_quenches=1000):
    """
    模拟 Quench 动力学中的雪崩 (Response Avalanche)。
    核心逻辑：统计对于微小的参数变化 delta_A，系统状态变化 delta_x 的分布。
    """

    
    # 2. 初始化系统状态
    # 我们需要一个非零的“外场”或“输入” y，使得 x 有非零的平衡位置
    # Ax = y  => x = A^-1 y
    y = 0.1*np.random.randn(N)
    x_current = np.linalg.solve(A, y)
    
    avalanche_sizes = []
    # durations=[]

    # # 时间轴 (用于积分)
    # t_eval = np.logspace(1, 10, 2000) # 覆盖快慢尺度
    
    # 3. 循环执行 Quench 实验
    for _ in range(num_quenches):
        # y = 0.1*np.random.randn(N)
        # x_current = np.linalg.solve(A, y)

        
        # --- A. 施加微扰 (The Quench) ---
        # 这种扰动模拟突触权重的微小漂移或学习更新
        # 强度必须很小，否则系统会彻底崩溃



        
        perturbation_strength = 1e-6
        R = np.random.randn(N, N)
        delta_A = perturbation_strength * (R + R.T) # 对称扰动
        
        # --- B. 计算新的平衡态 ---
        A_new = A + delta_A
        
        # 检查正定性 (可选，防止数值爆炸)
        # 在微小扰动下，Sloppy 系统的软模极易变负，这里简单处理：
        # 如果奇异，则跳过统计（或视为无穷大雪崩）


        x_new = np.linalg.solve(A_new, y)
            
        # --- C. 提取雪崩规模 (Avalanche Size) ---
        # 定义：状态向量在相空间中的位移距离
        dx = x_new - x_current
        size = np.linalg.norm(dx)
        
        avalanche_sizes.append(size)

        

        


        # --- D. 更新系统 ---
        # 让系统演化下去 (Cumulative Aging) 
        # 或者重置 A (Independent Samples)
        # 这里选择 Cumulative，模拟真实系统的连续演化
        A = A_new 
        x_current = x_new

        
        # try:
        #     x_new = np.linalg.solve(A_new, y)
            
        #     # --- C. 提取雪崩规模 (Avalanche Size) ---
        #     # 定义：状态向量在相空间中的位移距离
        #     dx = x_new - x_current
        #     size = np.linalg.norm(dx)
            
        #     avalanche_sizes.append(size)

        #     # --- D. 更新系统 ---
        #     # 让系统演化下去 (Cumulative Aging) 
        #     # 或者重置 A (Independent Samples)
        #     # 这里选择 Cumulative，模拟真实系统的连续演化
        #     A = A_new 
        #     x_current = x_new

        # except np.linalg.LinAlgError:
        #     # 矩阵奇异，意味着特征值触底 0，发生了灾难性雪崩
        #     pass
        

    return np.array(avalanche_sizes)

In [ ]:
sizes = simulate_quench_avalanche(A,model.N)

fig,ax=plt.subplots(figsize=(6, 4.5))
plot_power_law(sizes, r'$||\mathbf{dx}||$', ax, "Avalanche Size Distribution")
plt.savefig(result_dir+f'hetero_dx_powerlaw_{modes[2]}_N_{N}.png',bbox_inches='tight',dpi=300)

In [ ]:
# --- 运行 ---
A_shuffled = np.random.permutation(A.flatten()).reshape(A.shape)

sizes = simulate_quench_avalanche(A_shuffled,model.N)

fig,ax=plt.subplots(figsize=(6, 4.5))
plot_power_law(sizes, r'$||\mathbf{dx}||$', ax, "Avalanche Size Distribution")
plt.savefig(result_dir+f'hetero_dx_powerlaw_shuffled_{modes[2]}_N_{N}.png',bbox_inches='tight',dpi=300)